# Attention — content-based lookup over a sequence

> Tutorial pair for [`attention.py`](attention.py).

## 1. Intuition
Each position emits a **query** ("what am I looking for?"); every position offers
a **key** ("what do I contain?") and a **value** ("what I'll pass on"). A position's
output is a weighted average of all values, weighted by how well its query matches
each key. Unlike an RNN, this is a **direct, parallel** path between any two
positions — no information has to survive many recurrent steps.

## 2. Concept (the slide)
- **Scaled dot-product attention:** $\text{Attn}(Q,K,V)=\operatorname{softmax}\!\big(\tfrac{QK^\top}{\sqrt{d_k}}\big)V$.
- **Self-attention:** $Q,K,V$ all come from the same sequence.
- **Cross-attention:** $Q$ from one sequence, $K,V$ from another (decoder ↔ encoder).
- **Multi-head:** run $h$ attentions in parallel subspaces, concat, project — many
  relations at once.
- **Masking:** add $-\infty$ to forbidden scores (causal for autoregression, padding for variable lengths).

## 3. Math derivation

**The operation.** With $Q\in\mathbb R^{L_q\times d_k}$, $K\in\mathbb R^{L_k\times d_k}$,
$V\in\mathbb R^{L_k\times d_v}$:
$$S=\frac{QK^\top}{\sqrt{d_k}}\in\mathbb R^{L_q\times L_k},\quad
  A=\operatorname{softmax}(S)\ \text{(row-wise)},\quad
  \text{out}=AV.$$
Row $i$ of $A$ is a probability distribution over positions, so output $i$ is a
convex combination of the value vectors.

**Why divide by $\sqrt{d_k}$?** If query/key entries are independent with unit
variance, the dot product $q\!\cdot\!k=\sum_{j=1}^{d_k}q_jk_j$ has variance $d_k$.
Large logits push softmax into saturated regions where its Jacobian
$\operatorname{diag}(a)-aa^\top$ is tiny → **vanishing gradients**. Scaling by
$1/\sqrt{d_k}$ restores unit variance and healthy gradients.

**Multi-head.** Split $d_{\text{model}}$ into $h$ heads of size $d_k=d_{\text{model}}/h$:
$$\text{head}_i=\text{Attn}(QW_i^Q,KW_i^K,VW_i^V),\quad
  \text{MHA}=[\text{head}_1;\dots;\text{head}_h]\,W^O.$$
Each head can specialize (syntax, coreference, position…), and the cost matches a
single full-width attention.

**Masking.** Add an additive mask $M$ (0 keep, $-\infty$ block) *before* softmax:
$A=\operatorname{softmax}(S+M)$. A causal (upper-triangular $-\infty$) mask makes
position $i$ attend only to $\le i$ — required for left-to-right generation.

**Complexity.** $O(L^2 d)$ time and $O(L^2)$ memory — quadratic in sequence
length (the motivation for efficient-attention research).

## 4. NumPy implementation — the primitive + multi-head + masks

In [ ]:
# ===== actual implementation from attention.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def causal_mask(L):
    """Upper-triangular -inf so position i can't attend to j>i."""
    m = np.triu(np.ones((L, L)), k=1)
    return np.where(m == 1, -1e9, 0.0)

import torch

import torch.nn as nn

import torch.nn.functional as F

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    B, L, d_model, h = 2, 6, 16, 4
    x = np.random.randn(B, L, d_model).astype(np.float32)

    mha = MultiHeadAttentionNumPy(d_model, h)
    out = mha(x, x, x)                                # self-attention
    print(f"NumPy MHA self-attention out shape: {out.shape}")
    print(f"attention weights row sums (≈1): {mha.weights[0,0,0].sum():.3f}")

    # causal masking: position 0 should only attend to itself
    m = causal_mask(L)
    _ = mha(x, x, x, mask=m)
    upper = mha.weights[0, 0][np.triu_indices(L, k=1)]
    print(f"causal mask: max weight above diagonal = {upper.max():.2e} (≈0)")

    tor = MultiHeadAttentionTorch(d_model, h)
    ot = tor(torch.tensor(x), torch.tensor(x), torch.tensor(x))
    print(f"Torch MHA out shape: {tuple(ot.shape)}")

    # cross-attention: query length differs from key/value length
    q = np.random.randn(B, 3, d_model).astype(np.float32)
    oc = mha(q, x, x)
    print(f"cross-attention (Lq=3, Lk=6) out shape: {oc.shape}")


def scaled_dot_product_attention(Q, K, V, mask=None):
    r"""
    Attention(Q,K,V) = softmax( Q Kᵀ / sqrt(d_k) ) V

    Q:(..., Lq, d_k)  K:(..., Lk, d_k)  V:(..., Lk, d_v)
    mask: additive (0 keep, -inf block) broadcast to scores shape.
    Returns (output, attention_weights).
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)     # (..., Lq, Lk)
    if mask is not None:
        scores = scores + mask
    weights = softmax(scores, axis=-1)                     # rows sum to 1
    return weights @ V, weights


class MultiHeadAttentionNumPy:
    r"""
    Project Q,K,V into h heads, attend in each (d_k = d_model/h), concat, project.
        head_i = Attention(Q W_i^Q, K W_i^K, V W_i^V)
        MHA    = Concat(head_1..head_h) W^O
    Multiple heads let the model attend to different relations simultaneously.
    """

    def __init__(self, d_model, n_heads, seed=SEED):
        assert d_model % n_heads == 0
        self.d_model, self.h = d_model, n_heads
        self.d_k = d_model // n_heads
        rng = np.random.default_rng(seed)
        s = 1.0 / np.sqrt(d_model)
        self.Wq = rng.normal(0, s, (d_model, d_model))
        self.Wk = rng.normal(0, s, (d_model, d_model))
        self.Wv = rng.normal(0, s, (d_model, d_model))
        self.Wo = rng.normal(0, s, (d_model, d_model))

    def _split(self, x):                # (B,L,d) -> (B,h,L,d_k)
        B, L, _ = x.shape
        return x.reshape(B, L, self.h, self.d_k).transpose(0, 2, 1, 3)

    def _merge(self, x):                # (B,h,L,d_k) -> (B,L,d)
        B, h, L, d_k = x.shape
        return x.transpose(0, 2, 1, 3).reshape(B, L, h * d_k)

    def __call__(self, query, key, value, mask=None):
        Q = self._split(query @ self.Wq)
        K = self._split(key @ self.Wk)
        V = self._split(value @ self.Wv)
        out, self.weights = scaled_dot_product_attention(Q, K, V, mask)
        return self._merge(out) @ self.Wo

## 5. PyTorch implementation

In [ ]:
# ===== actual implementation from attention.py =====
class MultiHeadAttentionTorch(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_k).transpose(1, 2)

    def forward(self, q, k, v, mask=None):
        Q, K, V = self._split(self.Wq(q)), self._split(self.Wk(k)), self._split(self.Wv(v))
        scores = Q @ K.transpose(-1, -2) / self.d_k ** 0.5
        if mask is not None:
            scores = scores + mask
        self.weights = scores.softmax(-1)
        out = self.weights @ V
        B, h, L, d_k = out.shape
        out = out.transpose(1, 2).reshape(B, L, h * d_k)
        return self.Wo(out)

## 6. Run — self, causal, cross attention; shapes & mask check

In [ ]:
demo()

## 7. Visualization — an attention weight matrix

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import attention as M

np.random.seed(1)
L, d, h = 8, 16, 4
x = np.random.randn(1, L, d).astype("float32")
mha = M.MultiHeadAttentionNumPy(d, h)
mha(x, x, x, mask=M.causal_mask(L))               # causal self-attention

fig, axes = plt.subplots(1, h, figsize=(13, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(mha.weights[0, i], cmap="viridis")
    ax.set_title(f"head {i}"); ax.set_xlabel("key pos"); ax.set_ylabel("query pos")
plt.suptitle("Causal attention: lower-triangular (no peeking ahead)")
plt.tight_layout(); plt.show()

## 8. Takeaways
- Attention = softmax-weighted average of values, keyed by query·key similarity.
- The $1/\sqrt{d_k}$ scale is essential for stable softmax gradients.
- Multi-head = several relations in parallel; masking controls who sees whom.
- Stack attention + FFN + residual/LayerNorm → the
  **[Transformer](../architectures/transformer.ipynb)**.